# Hybrid RAG with Graph + Vector Retrieval on GCP (Neo4j + FAISS)

This notebook implements a hybrid Retrieval-Augmented Generation system 
deployed on Google Cloud Platform.

It combines:

- Vertex AI embeddings
- Vector similarity search
- Neo4j knowledge graph
- LLM generation via Vertex AI

The goal is to design a scalable hybrid retrieval system
that supports both semantic and relational reasoning.

## Motivation
Traditional RAG systems retrieve semantically similar chunks but struggle 
with relational queries. This implementation explores combining:

- Vector search (FAISS) → semantic similarity
- Knowledge graph (Neo4j) → relational structure
- LLM reasoning on merged context

## Goal

Build a hybrid RAG system over Kubernetes documentation that enables relational reasoning beyond pure semantic search.
The goal is to support structured reasoning over infrastructure concepts, not just retrieve text snippets.

## Architecture Overview

Pipeline (designed for cloud-native deployment):

1. Data ingestion
2. Embedding generation (Vertex AI)
3. Vector indexing
4. Graph construction (Neo4j)
5. Hybrid retrieval
6. LLM response generation (Vertex AI)

Minimal diagram:

<img src="mermaid-diagram.png" width="40%">

## Stack
- OpenAI / local embeddings
- FAISS
- Neo4j
- Python

## Why Kubernetes?

Kubernetes is a strong benchmark for hybrid RAG because:

- Its concepts are highly relational (e.g., Deployments manage ReplicaSets, which create Pods).
- Many questions require multi-hop reasoning.
- The official documentation is authoritative and well structured.
- It reflects real-world infrastructure complexity.

This makes it suitable for evaluating graph + vector retrieval strategies. This architecture is extensible to other infrastructure domains such as Terraform, Cloud APIs, or distributed systems documentation.

## 1. Environment Setup

In [1]:
# general libraries 
import os, time, warnings, logging, subprocess

# AI & graph libraries 
from langchain_google_vertexai import ChatVertexAI, HarmCategory, HarmBlockThreshold
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs import Neo4jGraph
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Newer LangChain version corrections
from pydantic import BaseModel, Field
from typing import List

# Silence GRPC and LangChain noise for cleaner logs
warnings.filterwarnings("ignore")
os.environ["GRPC_VERBOSITY"] = "ERROR"
logging.getLogger("grpc").setLevel(logging.ERROR)

## 2. Config section

In [2]:
# Google Cloud project where Vertex AI resources are provisioned.
# In production, this should be injected via environment variables
# or a secrets manager (not hardcoded).
PROJECT_ID = "rag-agent-488010"

# Deployment region for Vertex AI services.
# Keeping region explicit avoids latency inconsistencies and quota fragmentation.
REGION = "us-central1"

# LLM used for generation and structured ontology extraction.
# Gemini Flash is chosen for low latency and cost efficiency
# in multi-call pipelines (ontology induction, reranking, evaluation).
TARGET_MODEL = "gemini-2.0-flash-001"

# Embedding model for semantic retrieval.
# text-embedding-004 offers strong general-purpose semantic representations
# suitable for technical documentation.
EMBEDDING_MODEL = "text-embedding-004"


# -------------------------------
# Data Source Configuration
# -------------------------------

# Source repository used as corpus.
# Using official Kubernetes documentation ensures high-quality,
# domain-consistent ground truth.
REPO_URL = "https://github.com/kubernetes/website.git"

# Local clone path.
# In production, this would typically be replaced by a
# storage bucket (e.g., GCS) or a document ingestion pipeline.
REPO_PATH = "./k8s_docs_repo"


# -------------------------------
# Embedding Throughput Settings
# -------------------------------

# Number of texts per embedding batch request.
# Tuned to balance API quota limits, latency, and memory footprint.
BATCH_SIZE = 64

# Parallel workers for embedding generation.
# Thread-based parallelism is used to increase throughput
# while staying within Vertex AI rate limits.
MAX_WORKERS = 5

## 3. Data Ingestion

We ingest official Kubernetes documentation directly from GitHub.

In [3]:
def load_documents():
    """
    Clone the Kubernetes documentation repository (if not present)
    and load a filtered subset of Markdown files for indexing.

    Returns:
        List[Document]: Cleaned LangChain Document objects.
    """

    print("[INFO] Starting data ingestion...")

    # Clone repository locally if it does not already exist.
    # In production, this would typically be replaced by a storage bucket
    # or a managed ingestion pipeline.
    if not os.path.exists(REPO_PATH):
        print("[INFO] Cloning Kubernetes documentation repository...")
        subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

    all_docs = []

    # Load Markdown files from the conceptual documentation section.
    # We restrict ingestion to the "concepts" directory to reduce noise
    # and focus on high-level architectural explanations.
    print("[INFO] Loading Markdown files (.md)...")

    try:
        loader_md = DirectoryLoader(
            os.path.join(REPO_PATH, "content/en/docs/concepts"),
            glob="**/*.md",
            loader_cls=TextLoader,
            loader_kwargs={"autodetect_encoding": True}
        )

        docs_md = loader_md.load()
        print(f"[INFO] Loaded {len(docs_md)} Markdown documents.")
        all_docs.extend(docs_md)

    except Exception as e:
        print(f"[WARNING] Error while loading Markdown files: {e}")

    # Basic filtering:
    # - Remove very short documents (likely non-informative)
    # - Exclude license or meta files
    clean_docs = [
        d for d in all_docs
        if len(d.page_content) > 100
        and "LICENSE" not in d.metadata.get("source", "")
    ]

    print(f"[INFO] Valid documents after filtering: {len(clean_docs)}")

    return clean_docs


# Load corpus into memory (used for both vector indexing and graph construction)
global_docs = load_documents()

[INFO] Starting data ingestion...
[INFO] Loading Markdown files (.md)...
[INFO] Loaded 164 Markdown documents.
[INFO] Valid documents after filtering: 164


## 4. Text Chunking Strategy

We use:
- Small chunks for vector retrieval
- Large chunks for graph extraction

In [4]:
# -------------------------------
# Vector Retrieval Chunking
# -------------------------------
# Designed for embedding generation and FAISS indexing.
# Moderate chunk size balances semantic coherence and retrieval precision.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1100,       # Suitable size for embedding-based retrieval
    chunk_overlap=100,     # Overlap preserves cross-boundary context
    separators=[
        "\n## ",
        "\n### ",
        "\n#### ",
        "\n\n",
        "\n",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(global_docs)

print(f"[INFO] Vector chunks generated: {len(chunks)}")


# -------------------------------
# Graph Extraction Chunking
# -------------------------------
# Larger chunks are used for knowledge graph construction.
# This allows the LLM to observe complete conceptual structures
# and extract richer relationships in a single pass.

text_splitter_graph = RecursiveCharacterTextSplitter(
    chunk_size=80000,      # Large context for relational extraction
    chunk_overlap=200,     # Slight overlap to avoid boundary truncation
    separators=[
        "\n## ",
        "\n### ",
        "\n#### ",
        "\n\n",
        "\n",
        " ",
        ""
    ]
)

chunks_graph = text_splitter_graph.split_documents(global_docs)

print(f"[INFO] Graph chunks generated: {len(chunks_graph)}")

[INFO] Vector chunks generated: 2833
[INFO] Graph chunks generated: 164


## 5. Embedding Generation (Vertex AI) \& vector indexing (FAISS)

In [5]:
# --------------------------------------------------
# Embedding Generation (Vertex AI)
# --------------------------------------------------
# We generate semantic embeddings using a managed Vertex AI model.
# Parallel batching is used to increase throughput while respecting
# API quotas and latency constraints.

from langchain_google_vertexai import VertexAIEmbeddings
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_community.vectorstores import FAISS


# -------------------------------
# Embedding Model Configuration
# -------------------------------
# Managed embedding model from Vertex AI.
# Suitable for technical documentation and general semantic retrieval.

embeddings = VertexAIEmbeddings(
    model=EMBEDDING_MODEL,
    project=PROJECT_ID
)


# -------------------------------
# Prepare Text & Metadata
# -------------------------------

texts = [c.page_content for c in chunks]
metadatas = [c.metadata for c in chunks]

max_len = max(len(t) for t in texts)
print(f"[INFO] Maximum chunk length: {max_len} characters")


# -------------------------------
# Parallel Embedding Generation
# -------------------------------
# Strategy:
# - Split texts into batches
# - Process batches concurrently using thread pooling
# - Preserve original order for deterministic indexing
#
# This improves throughput while keeping embedding calls manageable.

def generate_embeddings_parallel(texts, batch_size=BATCH_SIZE, max_workers=MAX_WORKERS):
    """
    Generate embeddings in parallel batches.

    Args:
        texts (List[str]): Input texts to embed.
        batch_size (int): Number of texts per API call.
        max_workers (int): Number of parallel workers.

    Returns:
        List[List[float]]: Ordered embedding vectors.
    """

    ordered_embeddings = [None] * len(texts)
    futures = {}

    def embed_batch(batch_texts):
        return embeddings.embed_documents(batch_texts)

    print("[INFO] Generating embeddings in parallel...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            future = executor.submit(embed_batch, batch)
            futures[future] = i

        for future in as_completed(futures):
            start_idx = futures[future]
            batch_embs = future.result()
            ordered_embeddings[start_idx:start_idx + len(batch_embs)] = batch_embs

    return ordered_embeddings


ordered_embeddings = generate_embeddings_parallel(texts)


# -------------------------------
# Vector Store Construction
# -------------------------------
# FAISS is used for local prototyping.
# In production, this could be replaced by Vertex AI Vector Search
# or another distributed vector database.

text_embedding_pairs = list(zip(texts, ordered_embeddings))

vectorstore = FAISS.from_embeddings(
    text_embedding_pairs,
    embeddings,
    metadatas=metadatas,
    normalize_L2=True  # Improves cosine similarity behavior
)

print("[INFO] Embeddings generated and FAISS index created successfully.")

[INFO] Maximum chunk length: 1099 characters
[INFO] Generating embeddings in parallel...
[INFO] Embeddings generated and FAISS index created successfully.


# 6. Graph construction

### 6.1 Neo4j Connection

In [7]:
# --------------------------------------------------
# Neo4j Configuration (Knowledge Graph Layer)
# --------------------------------------------------
# Local Neo4j instance running via Docker.
# In production, this could be replaced by a managed service
# such as Neo4j AuraDB or a secured cloud deployment.

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "password"  # ⚠️ For demo purposes only (do not hardcode in production)


# -------------------------------
# Neo4j Connection Initialization
# -------------------------------
# We validate connectivity and ensure APOC procedures are available,
# since entity merging and graph refactoring depend on them.

try:
    graph = Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD
    )

    # Refresh schema to validate connection and detect APOC support
    graph.refresh_schema()

    print("[INFO] Neo4j connection established successfully (APOC detected).")

except Exception as e:
    print(f"[ERROR] Failed to connect to Neo4j: {e}")
    print("[HINT] Ensure the Docker container is running and APOC plugins are enabled.")

[INFO] Neo4j connection established successfully (APOC detected).


### 6.2 LLM-based Ontology Induction & Graph construction

We dynamically infer graph schema using Gemini.

In [10]:
# --------------------------------------------------
# Dynamic Ontology Discovery (LLM-driven)
# --------------------------------------------------
# Instead of manually defining a fixed schema, we infer
# a domain-specific ontology from the corpus using Gemini.
#
# This enables adaptive graph construction and reduces
# manual schema engineering effort.


import random


# Optional prefix to namespace dynamically created node types.
# Useful when mixing multiple graph construction phases.
PREFIX_DYNAMIC = ""

print("[INFO] Starting dynamic graph construction phase...")


# --------------------------------------------------
# Step A: Ontology Induction from Sample
# --------------------------------------------------

class OntologySchema(BaseModel):
    """
    Structured schema returned by the LLM for graph construction.
    """
    nodes: List[str] = Field(description="Graph node types")
    relationships: List[str] = Field(description="Relationship types")


print("[INFO] Inferring ontology from documentation sample...")

try:
    SAMPLE_SIZE = 12
    actual_sample_size = min(len(chunks_graph), SAMPLE_SIZE)
    random_sample = random.sample(chunks_graph, actual_sample_size)

    sample_text = "\n---\n".join([d.page_content for d in random_sample])

    print(f"[INFO] Analyzing {actual_sample_size} documents for ontology induction...")

    prompt = PromptTemplate(
        template="""
You are a Kubernetes domain architect.

From the following documentation excerpts:

{text}

Propose a concise ontology for a knowledge graph.

Return:
- Node types
- Relationship types

Avoid generic abstractions.
Prefer conceptual Kubernetes entities.
""",
        input_variables=["text"]
    )

    chain = prompt | llm.with_structured_output(OntologySchema)

    for attempt in range(3):
        try:
            res = chain.invoke({"text": sample_text})
            break
        except Exception as e:
            if attempt < 2:
                print(f"[WARNING] Ontology retry ({attempt+1}/3): {str(e)[:80]}")
                time.sleep(10)
            else:
                raise e

    dyn_nodes = [n.strip() for n in res.nodes]
    dyn_rels = [r.strip().upper().replace(" ", "_") for r in res.relationships]

    print(f"[INFO] Ontology discovered: {len(dyn_nodes)} node types, {len(dyn_rels)} relationship types.")

except Exception as e:
    print(f"[WARNING] Ontology induction failed: {e}")
    print("[INFO] Falling back to predefined schema.")

    dyn_nodes = [
        'Module', 'Class', 'Method', 'Function', 'State', 'Channel',
        'Node', 'Task', 'Interrupt', 'Config', 'Checkpoint',
        'Runnable', 'Value', 'Message', 'Edge', 'Policy', 'Tool'
    ]

    dyn_rels = [
        'CONTAINS', 'INHERITS_FROM', 'USES', 'WRITES_TO', 'READS_FROM',
        'UPDATES', 'SENDS_TO', 'TRIGGERS', 'DEFINES', 'HAS_ATTRIBUTE',
        'IMPLEMENTS', 'APPLIES_TO', 'HANDLES', 'MANAGES',
        'STORES', 'DEPENDS_ON', 'CALLS'
    ]


# --------------------------------------------------
# Step B: Graph Construction (Full Corpus)
# --------------------------------------------------
# Convert all chunks into graph entities using the
# dynamically inferred ontology.

llm = ChatVertexAI(
    model=TARGET_MODEL,
    temperature=0,  # 0 -> determinista, 1-> creativo 
    project=PROJECT_ID,
    location=REGION,
    max_retries=1,
    safety_settings={
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    }
)


llm_transformer_dyn = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=dyn_nodes,
    allowed_relationships=dyn_rels,
    strict_mode=False
)

total_chunks = len(chunks_graph)
print(f"[INFO] Starting graph population ({total_chunks} chunks)...")

for i, chunk in enumerate(chunks_graph, 1):

    try:
        graph_docs = llm_transformer_dyn.convert_to_graph_documents([chunk])

        # Apply optional namespace prefix
        for doc in graph_docs:
            for node in doc.nodes:
                node.type = PREFIX_DYNAMIC + node.type

        if graph_docs:
            graph.add_graph_documents(
                graph_docs,
                baseEntityLabel=True,
                include_source=True
            )

    except Exception as e:
        if "429" in str(e):
            print(f"[WARNING] Rate limit at chunk {i}. Waiting 30s...")
            time.sleep(30)
        else:
            print(f"[ERROR] Chunk {i} failed: {str(e)[:80]}")

    # Compact progress indicator
    print(f"[INFO] Chunk {i}/{total_chunks} processed", end="\r")

    time.sleep(1)

print("\n[INFO] Dynamic graph construction completed.")

[INFO] Starting dynamic graph construction phase...
[INFO] Inferring ontology from documentation sample...
[INFO] Analyzing 12 documents for ontology induction...
[INFO] Ontology discovered: 9 node types, 9 relationship types.
[INFO] Starting graph population (164 chunks)...
[INFO] Chunk 164/164 processed
[INFO] Dynamic graph construction completed.


### 6.3 Entity Resolution (APOC-based)

In [11]:
# --------------------------------------------------
# Entity Resolution & Graph Cleanup
# --------------------------------------------------
# After graph population, we perform:
# 1) Case-insensitive entity merging
# 2) Orphan node cleanup
#
# This improves graph consistency and reduces redundancy
# caused by LLM-based extraction variability.

print("[INFO] Starting entity resolution phase...")


# --------------------------------------------------
# Step 1: Case-Insensitive Entity Merge
# --------------------------------------------------
# Merge nodes that share the same ID ignoring case differences.
# Requires APOC procedures.

merge_query = """
MATCH (n:__Entity__)
WITH toLower(n.id) AS lowerId, collect(n) AS nodes
WHERE size(nodes) > 1
CALL apoc.refactor.mergeNodes(nodes, {properties: 'combine', mergeRels: true})
YIELD node
RETURN count(node) as mergedCount
"""

try:
    res = graph.query(merge_query)
    merged_count = res[0]["mergedCount"] if res else 0
    print(f"[INFO] Entities merged (case-insensitive): {merged_count}")

except Exception as e:
    print(f"[WARNING] Entity merge failed: {e}")
    print("[INFO] APOC Full is required for advanced merge operations.")


# --------------------------------------------------
# Step 2: Orphan Cleanup
# --------------------------------------------------
# Remove nodes without valid identifiers.
# These may appear due to partial or malformed extraction.

cleanup_query = """
MATCH (n:__Entity__)
WHERE n.id IS NULL
DETACH DELETE n
"""

graph.query(cleanup_query)

print("[INFO] Graph cleanup completed.")

[INFO] Starting entity resolution phase...
[INFO] Entities merged (case-insensitive): 0
[INFO] Graph cleanup completed.


## 7. Hybrid Retrieval + Generation

In [14]:
# --------------------------------------------------
# Graph-Based Retrieval Layer
# --------------------------------------------------
# This module extracts structured evidence from the knowledge graph.
# It supports:
#   - Entity extraction from user queries
#   - Single-hop graph lookup
#   - Multi-hop reasoning paths
#
# The goal is to provide relational context that complements
# vector-based semantic retrieval.


# --------------------------------------------------
# Entity Extraction from Query
# --------------------------------------------------
# We use the LLM to detect Kubernetes entities mentioned
# in the user question. These entities act as graph anchors.

def extract_entities(query: str):
    """
    Extract Kubernetes entities referenced in a user question.

    Returns:
        List[str]: Candidate entity identifiers.
    """
    prompt = f"""
Extract Kubernetes entities mentioned in the question.
Return a comma-separated list.

Question:
{query}
"""
    res = llm.invoke(prompt)
    return [e.strip() for e in res.content.split(",") if e.strip()]


# --------------------------------------------------
# Base Graph Lookup (Single-Hop)
# --------------------------------------------------
# Retrieves direct relationships connected to extracted entities.

def graph_lookup(entities, limit=30):
    """
    Retrieve single-hop relationships for given entities.
    """

    if not entities:
        return ""

    cypher_query = """
    MATCH (a)-[r]->(b)
    WHERE 
        (a.id IN $entities OR b.id IN $entities)
        AND NOT a.id =~ '^[0-9a-f]{32}$'
        AND NOT b.id =~ '^[0-9a-f]{32}$'
        AND a.id <> b.id
    RETURN 
        a.id   AS source,
        type(r) AS relation,
        b.id   AS target
    LIMIT $limit
    """

    results = graph.query(
        cypher_query,
        {"entities": entities, "limit": limit}
    )

    if not results:
        return ""

    return "\n".join(
        f"{r['source']} - {r['relation']} - {r['target']}"
        for r in results
    )


# --------------------------------------------------
# Relation Filtering Strategy
# --------------------------------------------------
# We distinguish between:
#   - Control / causal relations (high reasoning value)
#   - Secondary configuration relations
#
# This improves reasoning quality in multi-hop traversal.

SINGLEHOP_RELATIONS = [
    "MANAGES",
    "CREATES",
    "RUNS_ON",
    "USES",
    "APPLIES_TO",
    "BELONGS_TO"
]

MULTIHOP_RELATIONS = [
    "MANAGES",
    "CREATES",
    "RUNS_ON",
    "USES",
    "APPLIES_TO"
]

ALLOWED_ROOTS = [
    "Deployment",
    "ReplicaSet",
    "Service",
    "Node",
    "Namespace",
    "Controller",
    "Workload"
]


# --------------------------------------------------
# Multi-Hop Graph Lookup
# --------------------------------------------------
# This function retrieves:
#   1) Single-hop control relations
#   2) Two-hop reasoning paths
#
# Multi-hop traversal enables causal reasoning such as:
# Deployment → ReplicaSet → Pod

single_hop_query = """
    MATCH (a)-[r]->(b)
    WHERE 
        (a.id IN $entities OR b.id IN $entities)
        AND type(r) IN $allowed_rels
        AND NOT a.id =~ '^[0-9a-f]{32}$'
        AND NOT b.id =~ '^[0-9a-f]{32}$'
        AND a.id <> b.id
    RETURN 
        a.id   AS source,
        type(r) AS relation,
        b.id   AS target
    LIMIT $limit
    """

multi_hop_query = """
    MATCH (a)-[r1]->(b)-[r2]->(c)
    WHERE 
        a.id IN $entities
        AND a.id IN $allowed_roots
        AND b.id IN ['ReplicaSet','Pod','Deployment']
        AND type(r1) IN $multihop_rels
        AND type(r2) IN $multihop_rels
        AND NOT a.id =~ '^[0-9a-f]{32}$'
        AND NOT b.id =~ '^[0-9a-f]{32}$'
        AND NOT c.id =~ '^[0-9a-f]{32}$'
        AND a.id <> b.id
        AND b.id <> c.id
    RETURN 
        a.id AS source,
        type(r1) AS relation1,
        b.id AS middle,
        type(r2) AS relation2,
        c.id AS target
    LIMIT $limit
    """

def graph_lookup_multihop(entities, limit=20):

    if not entities:
        return ""


    params = {
        "entities": entities,
        "limit": limit,
        "allowed_rels": SINGLEHOP_RELATIONS,
        "multihop_rels": MULTIHOP_RELATIONS,
        "allowed_roots": ALLOWED_ROOTS
    }

    single_results = graph.query(single_hop_query, params)
    multi_results  = graph.query(multi_hop_query, params)

    lines = []

    for r in single_results:
        lines.append(
            f"{r['source']} - {r['relation']} - {r['target']}"
        )

    for r in multi_results:
        lines.append(
            f"{r['source']} - {r['relation1']} - "
            f"{r['middle']} - {r['relation2']} - {r['target']}"
        )

    return "\n".join(lines)


# --------------------------------------------------
# Explainable Multi-Hop Lookup
# --------------------------------------------------
# Returns structured reasoning paths to enable
# step-by-step explanation in the final answer.

def graph_exp_lookup_multihop(entities, limit=20):

    if not entities:
        return {"single": [], "multi": []}

    params = {
        "entities": entities,
        "limit": limit,
        "allowed_rels": SINGLEHOP_RELATIONS,
        "multihop_rels": MULTIHOP_RELATIONS,
        "allowed_roots": ALLOWED_ROOTS
    }

    single_results = graph.query(single_hop_query, params)
    multi_results  = graph.query(multi_hop_query, params)

    single_facts = [
        f"{r['source']} {r['relation']} {r['target']}"
        for r in single_results
    ]

    multi_paths = [
        [
            f"{r['source']} {r['relation1']} {r['middle']}",
            f"{r['middle']} {r['relation2']} {r['target']}"
        ]
        for r in multi_results
    ]

    return {
        "single": single_facts,
        "multi": multi_paths
    }


## 8. Hybrid Re-ranked Retrieval + Graph generation

In [15]:
# --------------------------------------------------
# Hybrid Retrieval-Oriented Generation (RAG Layer)
# --------------------------------------------------
# This module orchestrates:
#   1) Vector retrieval (semantic search)
#   2) LLM-based re-ranking
#   3) Graph-based relational retrieval
#   4) Context assembly
#   5) Final answer generation
#
# The goal is to combine semantic similarity and
# structured reasoning into a single grounded response.


# --------------------------------------------------
# Vector Retriever Configuration
# --------------------------------------------------
# We increase k to improve recall before re-ranking.
# High recall + LLM re-ranking typically improves final relevance.

retriever = vectorstore.as_retriever(search_kwargs={"k": 30})
MAX_CONTEXT_CHARS = 12000


# --------------------------------------------------
# RAG Prompt Template
# --------------------------------------------------
# The prompt explicitly guides:
#   - Grounded answering
#   - Preference for causal/control relations
#   - Structured reasoning from graph paths

rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a Kubernetes documentation expert.

Use only the provided context.
Prefer high-level descriptive documentation over implementation details.
If multiple pieces of context exist, choose those that define purpose and behavior.
Cite sources when relevant.

GRAPH PATH INTERPRETATION RULES:

- Prefer CONTROL or CAUSATION paths:
  MANAGES, CREATES, RUNS_ON, APPLIES_TO.
- Treat configuration/resource relations as secondary.
- For "how" or "why" questions, prioritize causal chains.

If the answer is not present, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)


# --------------------------------------------------
# LLM-Based Re-ranking
# --------------------------------------------------
# We use the LLM as a relevance judge to refine
# the top-k retrieved documents.

def rerank_documents(query, docs, top_n=5):
    """
    Re-rank retrieved documents using LLM scoring.
    """

    scored_docs = []

    for d in docs:
        scoring_prompt = f"""
You are a relevance judge.

Question:
{query}

Passage:
{d.page_content}

Score from 1 to 10 how well this passage answers the question.
Only output a number.
"""
        try:
            score_text = llm.invoke(scoring_prompt).content.strip()
            score = int(score_text)
        except:
            score = 0

        scored_docs.append((score, d))

    scored_docs.sort(key=lambda x: x[0], reverse=True)
    return [d for score, d in scored_docs[:top_n]]


# --------------------------------------------------
# Context Builder
# --------------------------------------------------
# Concatenates retrieved documents while enforcing
# a maximum token/character budget.

def build_context(docs, max_chars=MAX_CONTEXT_CHARS):

    context_parts = []
    total = 0

    for d in docs:
        block = f"SOURCE: {d.metadata.get('source')}\n{d.page_content}"

        if total + len(block) > max_chars:
            remaining = max_chars - total
            if remaining > 0:
                context_parts.append(block[:remaining])
            break

        context_parts.append(block)
        total += len(block)

    return "\n\n".join(context_parts)


# --------------------------------------------------
# Hybrid Answer Function
# --------------------------------------------------
# Combines:
#   - Semantic retrieval
#   - Graph relational lookup
#   - Prompted generation

def ask(query: str):
    """
    Generate answer using hybrid vector + graph retrieval.
    """

    # ---- Vector Retrieval ----
    docs = retriever.invoke(query)
    docs = rerank_documents(query, docs)

    filtered_docs = [
        d for d in docs
        if "docs" in d.metadata.get("source", "").lower()
        or "readme" in d.metadata.get("source", "").lower()
    ]

    if not filtered_docs:
        filtered_docs = docs

    vector_context = build_context(filtered_docs)

    # ---- Graph Retrieval ----
    entities = extract_entities(query)
    graph_facts = graph_lookup_multihop(entities)

    # ---- Hybrid Context Assembly ----
    context = f"""
GRAPH FACTS:
{graph_facts}

DOCUMENTS:
{vector_context}
"""

    final_prompt = rag_prompt.format(
        context=context,
        question=query
    )

    answer = llm.invoke(final_prompt).content

    return answer, vector_context, graph_facts


# --------------------------------------------------
# Explainable Hybrid Answer
# --------------------------------------------------
# Returns structured graph paths for step-by-step reasoning.

def ask_explain(query: str):

    # ---- Vector Retrieval ----
    docs = retriever.invoke(query)
    docs = rerank_documents(query, docs)

    filtered_docs = [
        d for d in docs
        if "docs" in d.metadata.get("source", "").lower()
        or "readme" in d.metadata.get("source", "").lower()
    ]

    if not filtered_docs:
        filtered_docs = docs

    vector_context = build_context(filtered_docs)

    # ---- Graph Retrieval (Explainable) ----
    entities = extract_entities(query)
    graph_data = graph_exp_lookup_multihop(entities)

    graph_parts = []

    if graph_data["multi"]:
        graph_parts.append("GRAPH PATHS:")
        for i, path in enumerate(graph_data["multi"], 1):
            graph_parts.append(f"PATH {i}:")
            for step in path:
                graph_parts.append(f"- {step}")

    if graph_data["single"]:
        graph_parts.append("\nGRAPH FACTS:")
        for fact in graph_data["single"]:
            graph_parts.append(f"- {fact}")

    graph_context = "\n".join(graph_parts)

    context = f"""
{graph_context}

DOCUMENTS:
{vector_context}
"""

    explanation_instruction = """
If GRAPH PATHS are present:
- Explain the reasoning chain step by step.
- Make causal structure explicit.
"""

    final_prompt = rag_prompt.format(
        context=context,
        question=query
    ) + explanation_instruction

    answer = llm.invoke(final_prompt).content

    return answer, vector_context, graph_data

## 9. Lets try the system

We now test the system with a causal question that requires 
multi-hop reasoning over Kubernetes abstractions:

**Question:** 
 
"How does a Deployment update Pods?"

This question cannot be answered correctly with simple keyword matching.
It requires understanding the control hierarchy:

Deployment → ReplicaSet → Pod

We evaluate whether the system:
1. Retrieves relevant documentation.
2. Extracts correct graph relationships.
3. Produces a causally structured explanation.

In [18]:
answer, vector_context, graph_data = ask_explain(
    "How does a Deployment update Pods?"
)

print(answer)

A Deployment updates Pods by managing ReplicaSets and scaling them. When a Deployment is updated, it creates a new ReplicaSet and scales it up, while simultaneously scaling down the old ReplicaSet. This ensures that a specific number of Pods are available during the update (at least 75% by default) and that the number of Pods created doesn't exceed a certain limit (at most 125% by default) [k8s_docs_repo/content/en/docs/concepts/workloads/controllers/deployment.md]. The Deployment controller manages this process, creating new ReplicaSets and scaling down old ones until the desired state is achieved [k8s_docs_repo/content/en/docs/concepts/workloads/controllers/deployment.md].



## Why the Answer Is Correct

The generated response demonstrates correct hybrid reasoning:

### a) Correct Control Chain

The answer explains that:

- A Deployment creates a new ReplicaSet.
- The new ReplicaSet scales up Pods.
- The old ReplicaSet is scaled down.
- The process maintains availability constraints (75% / 125%).

This matches the expected causal path:

Deployment ── MANAGES ──► ReplicaSet ── CREATES ──► Pod

The explanation reflects control relationships rather than merely listing related objects.


### b) Multi-Hop Reasoning

Updating Pods is not a direct action performed by the Deployment.
Instead, it happens indirectly via ReplicaSets.

The system correctly infers this two-step process:
- Deployment updates ReplicaSets.
- ReplicaSets control Pods.

This confirms that graph-based multi-hop traversal is working as intended.


### c) Groundedness in Documentation

The response cites the official Kubernetes documentation:

k8s_docs_repo/content/en/docs/concepts/workloads/controllers/deployment.md

This indicates:
- Vector retrieval surfaced the correct source.
- The answer remains grounded in authoritative documentation.
- No hallucinated mechanics were introduced.


### d) Structured Explanation

The answer:
- Describes scaling behavior.
- Mentions availability guarantees.
- Explains desired state reconciliation.
- Identifies the Deployment controller as the actor.

This aligns with Kubernetes' reconciliation model.

Lets check the graph path:

In [19]:
print("Graph Evidence Used:\n")
print(graph_data)

Graph Evidence Used:

{'single': ['Configmap BELONGS_TO Pod', 'Configmap RUNS_ON Pod', 'Configmap USES Pod', 'Configmap APPLIES_TO Pod', 'Secret BELONGS_TO Pod', 'Secret USES Pod', 'Service BELONGS_TO Pod', 'Service MANAGES Pod', 'Service USES Pod', 'Service USES Deployment', 'Pod BELONGS_TO Namespace', 'Pod BELONGS_TO Replicaset', 'Pod BELONGS_TO Kubernetes Api', 'Pod BELONGS_TO Kubernetes Objects', 'Pod BELONGS_TO Secret', 'Pod BELONGS_TO Deployment', 'Pod BELONGS_TO Statefulset', 'Pod BELONGS_TO Test', 'Pod BELONGS_TO Service', "Pod BELONGS_TO The Pod'S Serviceaccount"], 'multi': [['Deployment MANAGES Pod', 'Pod MANAGES Job Objects'], ['Deployment MANAGES Pod', 'Pod RUNS_ON Kubelet'], ['Deployment MANAGES Pod', 'Pod RUNS_ON Nginx'], ['Deployment MANAGES Pod', 'Pod RUNS_ON Node'], ['Deployment MANAGES Pod', 'Pod USES Environment Variable'], ['Deployment MANAGES Pod', 'Pod USES Volume'], ['Deployment MANAGES Pod', 'Pod USES Resourceclaim'], ['Deployment MANAGES Pod', 'Pod USES Cpu'], 

The extracted graph paths confirm that:

- Deployment MANAGES ReplicaSet
- ReplicaSet CREATES Pods

This structured evidence supports the causal explanation 
given in the final answer.

## 10. Evaluation Framework

We evaluate:

- Retrieval relevance
- Groundedness
- QA accuracy
- Answer quality

This test suite is ready for you to test and change anything that comes to mind, it also saves all the info so you can monitor how your changes affects the performance :).

In [21]:
# --------------------------------------------------
# Evaluation Suite
# --------------------------------------------------
# We evaluate the hybrid RAG system across:
#   - Definition questions
#   - Conceptual comparisons
#   - Control-plane reasoning
#
# The evaluation measures:
#   1) Retrieval relevance
#   2) Groundedness
#   3) QA accuracy
#   4) Answer quality
#
# Scoring is performed using an LLM-as-judge strategy.


TEST_QUESTIONS = [
    "What is a Pod in Kubernetes?",
    "What is a Deployment?",
    "What is the difference between a Pod and a Deployment?",
    "What is a Service in Kubernetes?",
    "What is the role of the kubelet?",
    "What is a ReplicaSet?",
    "What is the difference between a Deployment and a DaemonSet?"
]

In [54]:
import json
import re
from datetime import datetime


# --------------------------------------------------
# Hybrid Inference (Test Mode)
# --------------------------------------------------
# Generates answer + full context for evaluation.

def ask_test(query: str):
    """
    Generate answer using hybrid retrieval.
    Returns:
        - answer
        - vector_context (ONLY textual retrieval)
        - full_context (vector + graph, used for generation only)
    """

    # ---- Vector Retrieval ----
    docs = retriever.invoke(query)
    docs = rerank_documents(query, docs)

    filtered_docs = [
        d for d in docs
        if "docs" in d.metadata.get("source", "").lower()
        or "readme" in d.metadata.get("source", "").lower()
    ]

    if not filtered_docs:
        filtered_docs = docs

    vector_context = build_context(filtered_docs)

    # ---- Graph Retrieval ----
    entities = extract_entities(query)
    graph_facts = graph_lookup_multihop(entities)

    # ---- Hybrid Context (used ONLY for generation) ----
    full_context = f"""
GRAPH FACTS:
{graph_facts}

DOCUMENTS:
{vector_context}
"""

    final_prompt = rag_prompt.format(
        context=full_context,
        question=query
    )

    answer = llm.invoke(final_prompt).content

    return answer, vector_context


# --------------------------------------------------
# LLM-Based Evaluation (Judge)
# --------------------------------------------------
# Evaluates groundedness, relevance and QA accuracy.

def evaluate_answer(question: str, context: str, answer: str):

    eval_prompt = f"""
    You are a very strict evaluator for a Retrieval-Augmented Generation (RAG) system.
    Be very critical and skeptical.

    Only give 5 if the criterion is perfectly satisfied. Rarely give 5 score. 
    If the answer avoids the question or is indirect, penalize heavily.
    Penalize:
    - Generic answers
    - Indirect answers
    - Overly verbose answers
    - Missing causal steps
    - Claims not explicitly supported by context

    Question:
    {question}

    Context:
    {context}

    Answer:
    {answer}

    Evaluate from 1 (poor) to 5 (excellent) using 2 decimals:

    1. Retrieval Relevance:
    Is the retrieved context directly useful? 

    2. Groundedness:
    Are all important claims supported by the context?

    3. Question Answering Accuracy:
    Does the answer directly and correctly address the user's question?

    4. Answer Quality:
    Is the answer clear, complete, and well structured?

    Give also a feedback on the answer

    Return ONLY a JSON object:

    {{
    "retrieval_relevance": number,
    "groundedness": number,
    "qa_accuracy": number,
    "answer_quality": number,
    "feedback": string
    }}
    """
    # change the judge llm to avoid self-confirmation
    judge_llm = ChatVertexAI(
    model="gemini-2.5-flash",
    project=PROJECT_ID,
    location=REGION,
    temperature=0
    )

    return judge_llm.invoke(eval_prompt).content


# --------------------------------------------------
# Robust JSON Parsing
# --------------------------------------------------

def parse_metrics(raw_text):

    try:
        return json.loads(raw_text)

    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw_text, re.S)
        if match:
            return json.loads(match.group())

        raise ValueError("Failed to parse metrics JSON.")


# --------------------------------------------------
# Test Suite Execution
# --------------------------------------------------

def run_test_suite():

    results = []

    print("[INFO] Starting evaluation suite...")

    for i, question in enumerate(TEST_QUESTIONS, 1):

        print(f"[INFO] ({i}/{len(TEST_QUESTIONS)}) Evaluating: {question}")

        answer, context = ask_test(question)

        raw_metrics = evaluate_answer(question, context, answer)
        metrics = parse_metrics(raw_metrics)

        results.append({
            "question": question,
            "answer": answer,
            "metrics": metrics
        })

        print(f"[INFO] Metrics: {metrics}")

    print("[INFO] Evaluation completed.")

    return results


# --------------------------------------------------
# Metric Aggregation
# --------------------------------------------------

def aggregate_results(results):

    agg = {
        "retrieval_relevance": 0,
        "groundedness": 0,
        "qa_accuracy": 0,
        "answer_quality": 0
    }

    for r in results:
        for k in agg:
            agg[k] += r["metrics"][k]

    n = len(results)

    return {k: round(v / n, 2) for k, v in agg.items()}


# --------------------------------------------------
# Experiment Persistence
# --------------------------------------------------

def save_results(results, averages, tag="experiment"):

    payload = {
        "tag": tag,
        "timestamp": datetime.utcnow().isoformat(),
        "averages": averages,
        "results": results
    }

    filename = f"rag_eval_{tag}.json"

    with open(filename, "w") as f:
        json.dump(payload, f, indent=2)

    print(f"[INFO] Results saved to {filename}")

In [55]:
results = run_test_suite()
averages = aggregate_results(results)

print("\n[SUMMARY] Average Scores:")
print(averages)

save_results(results, averages, tag="hybrid_k8s_v1")

[INFO] Starting evaluation suite...
[INFO] (1/7) Evaluating: What is a Pod in Kubernetes?
[INFO] Metrics: {'retrieval_relevance': 5.0, 'groundedness': 5.0, 'qa_accuracy': 4.5, 'answer_quality': 4.0, 'feedback': "The answer is highly accurate and directly addresses the question, with every claim perfectly grounded in the provided context. It correctly identifies a Pod as the smallest deployable unit, a group of containers with shared resources, and emphasizes that Kubernetes manages Pods directly. To achieve a perfect score for accuracy and quality, the answer could have integrated the concept of containers within a Pod being 'co-located and co-scheduled' or forming an 'application-specific logical host' more explicitly, as these are fundamental characteristics mentioned in the context. While the current answer is excellent, these additions would provide a slightly more complete and synthesized definition without adding verbosity."}
[INFO] (2/7) Evaluating: What is a Deployment?
[INFO] 

# Final evaluation

## Limitations

- FAISS is local (not distributed)
- No graph embeddings
- No caching
- LLM-based reranking increases latency
- Ontology discovery depends on sample quality

## Production Considerations

- Wrap as FastAPI service
- Persist FAISS index
- Use Vertex AI Vector Search for scaling
- Move Neo4j to managed AuraDB
- Add caching layer
- Add monitoring (Cloud Logging)
- Add cost tracking